<a href="https://colab.research.google.com/github/ian-menachery/ECON3916-Statistical-Machine-Learning/blob/main/%5BLab_9%5D_Causal_Inference_and_Propensity_Score_Matching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

# Load your dataset here (ensure lalonde.csv is uploaded to Colab or linked)
df = pd.read_csv('lalonde.csv')

# Naive Comparison
treated_mean = df[df['treat'] == 1]['re78'].mean()
control_mean = df[df['treat'] == 0]['re78'].mean()
naive_diff = treated_mean - control_mean

print(f"Naive Difference in Means: ${naive_diff:,.2f}")
# Expected Result: -$635.03 (approximate, depending on your exact lalonde subset)

Naive Difference in Means: $-635.03


In [8]:
# Define covariates
# (Adjust these column names if your lalonde.csv uses different headers)
covariates = ['age', 'educ', 'black', 'hispan', 'married', 'nodegree', 're75', 're78']

X = df[covariates]
y = df['treat']

# Fit Propensity Model
logit = LogisticRegression(solver='liblinear')
logit.fit(X, y)

# Generate Scores (The probability of being treated)
df['pscore'] = logit.predict_proba(X)[:, 1]

In [9]:
from sklearn.neighbors import NearestNeighbors

# Separate groups
treated = df[df.treat==1]
control = df[df.treat==0]

# Fit NN on Control scores (Looking for 1 closest match)
nbrs = NearestNeighbors(n_neighbors=1).fit(control[['pscore']])

# Find matches for Treated scores
distances, indices = nbrs.kneighbors(treated[['pscore']])

# Extract the matched control units
matched_control = control.iloc[indices.flatten()]

# Construct Matched DataFrame
matched_df = pd.concat([treated, matched_control])

In [10]:
from scipy import stats

# T-test on raw data
diff = treated['re78'].mean() - control['re78'].mean()
t_stat, p_val = stats.ttest_ind(treated['re78'], control['re78'])

print("--- BEFORE MATCHING ---")
print(f"Raw Effect (Difference): ${diff:,.2f}")
print(f"P-value: {p_val:.4f}\n")


# Isolate the matched outcomes
matched_treated = matched_df[matched_df.treat==1]['re78']
matched_control = matched_df[matched_df.treat==0]['re78']

# Estimate the causal effect (T-test on matched data)
matched_diff = matched_treated.mean() - matched_control.mean()
matched_t_stat, matched_p_val = stats.ttest_ind(matched_treated, matched_control)

print("--- AFTER MATCHING ---")
print(f"Recovered Effect (Matched Difference): ${matched_diff:,.2f}")
print(f"P-value: {matched_p_val:.4f}")

--- BEFORE MATCHING ---
Raw Effect (Difference): $-635.03
P-value: 0.3342

--- AFTER MATCHING ---
Recovered Effect (Matched Difference): $1,850.03
P-value: 0.0109
